### Test Summary

This notebook demonstrates comprehensive testing of the CSV File Inspector script with various options:

1. Basic validation (non-existent file test)
2. Default information output
3. Column uniqueness report
4. Field listing
5. Sample rows display
6. Statistical analysis of fields
7. Unique values for specific columns
8. Output format tests (txt, markdown, LaTeX)
9. Column type separation
10. Grouped summary tables with different aggregation methods
11. Combined options for comprehensive analysis

The tests verify that all major functionalities of the script work correctly and produce the expected output formats.

### Dependencies

In [ ]:
import subprocess
import os

from library import ROOT, is_valid_file, is_valid_directory

### Set constants

In [ ]:
SCRIPT_PATH = os.path.join(ROOT, "core/src/utils/inspect_csv_file.py")
if not is_valid_file(SCRIPT_PATH):
    raise ValueError("Invalid Python script path.")

MOCK_CSV_PATH = os.path.join(
    ROOT,
    "core/tests/unit_tests/mock_data/valid_csv_files",
    "KL_several_m_varying_EpsCG_and_EpsMSCG_processed_parameter_values.csv",
)
if not is_valid_file(MOCK_CSV_PATH):
    raise ValueError("Invalid mock .csv script path.")

# Create output directory if it doesn't exist
OUTPUT_DIRECTORY = os.path.join(
    ROOT,
    "core/tests/integration_tests/test_inspect_csv_file_output",
)
if not is_valid_directory(OUTPUT_DIRECTORY):
    raise ValueError("Invalid mock .csv script path.")

### Helper function to run commands

In [ ]:
def run_command(cmd, description):
    """Helper function to run a command and display output with a description"""
    print(f"\n{'='*80}")
    print(f"TESTING: {description}")
    print(f"COMMAND: {' '.join(cmd)}")
    print(f"{'='*80}\n")

    # Run the command and capture real-time output
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True
    )

    # Print output in real-time
    for line in process.stdout:
        print(line, end="")

    # Print any errors
    stdout, stderr = process.communicate()
    if stderr:
        print("\nERRORS:")
        print(stderr)

    # Check exit code
    print()
    print("-" * 50)
    if process.returncode == 0:
        print(f"✅ Test '{description}' passed!")
    else:
        print(f"❌ Test '{description}' failed!")
    print("-" * 50)

### Test non-existent .csv file

In [ ]:
# Define paths
nonexistent_csv_path = "/path/to/nonexistent/file.csv"  # This path doesn't exist

# Run the script with the non-existent file
cmd = ["python", SCRIPT_PATH, "-csv", nonexistent_csv_path]

# Execute the command
result = subprocess.run(cmd, capture_output=True, text=True)

# Check if the error was properly returned
print(f"Return code: {result.returncode}")
print("\nSTDERR:")
print(result.stderr)
print("\nSTDOUT:")
print(result.stdout)

# Verify the script exited with a non-zero status for the error case
assert (
    result.returncode != 0
), "Script should fail with non-zero exit code for non-existent file"
print("✅ Test passed: Script correctly detected non-existent file")

### Test default information output:

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "-v",
    "--output_filename",
    "test_default_information_output",
]

run_command(cmd, "Display default information output")

### Test uniqueness report

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--uniqueness_report",
    "--output_filename",
    "test_uniqueness_report",
]

run_command(cmd, "Show uniqueness report")

### Test list fields

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--list_fields",
    "--output_filename",
    "test_list_fields",
]

run_command(cmd, "Show list of fields")

### Test sample rows

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--sample_rows",
    "5",  # Show 5 sample rows
    "--output_filename",
    "test_sample_rows",
]

run_command(cmd, "Sample rows display")

### Test field statistics

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--field_statistics",
    "--output_filename",
    "test_field_statistics",
]

run_command(cmd, "Field statistics")

### Test show unique values for a specific column

In [ ]:
# First, let's list the fields to choose one for unique values test
fields_cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "--list_fields",
]

print("Fetching column names to use for unique values test:\n")
fields_process = subprocess.run(fields_cmd, capture_output=True, text=True)
print(fields_process.stdout)

# Let's assume we've identified a column name from the output above
# Replace 'column_name' with an actual column name from your CSV
column_name = "m"  # Update this with an actual column from your data

cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--show_unique_values",
    column_name,
    "--output_filename",
    "test_unique_values",
]

run_command(cmd, f"Show unique values for column '{column_name}'")

### Test different output formats (Markdown)

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--output_format",
    "md",  # Use markdown format
    "--sample_rows",
    "3",
    "--field_statistics",
    "--output_filename",
    "test_markdown_output",
]

run_command(cmd, "Output in Markdown format")

# Display the content of the generated markdown file
md_file = os.path.join(OUTPUT_DIRECTORY, "test_markdown_output_summary.md")
if os.path.exists(md_file):
    print("\nGenerated Markdown file contents:")
    print("-" * 50)
    with open(md_file, "r") as f:
        print(f.read())
else:
    print(f"\nWarning: Could not find expected markdown file at {md_file}")

### Test different output formats (LaTeX)

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--output_format",
    "tex",  # Use LaTeX format
    "--sample_rows",
    "3",
    "--field_statistics",
    "--output_filename",
    "test_latex_output",
]

run_command(cmd, "Output in LaTeX format")

# Display the content of the generated LaTeX file
tex_file = os.path.join(OUTPUT_DIRECTORY, "test_latex_output_summary.tex")
if os.path.exists(tex_file):
    print("\nGenerated LaTeX file contents (first 20 lines):")
    print("-" * 50)
    with open(tex_file, "r") as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if i < 20:  # Show just the first 20 lines to avoid overwhelming output
                print(line, end="")
            else:
                print("... (more content omitted)")
                break
else:
    print(f"\nWarning: Could not find expected LaTeX file at {tex_file}")

### Test separate by type

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--uniqueness_report",
    "--separate_by_type",
    "--output_filename",
    "test_separate_by_type",
]

run_command(cmd, "Separate columns by type in uniqueness report")

### Test grouped summary tables

In [ ]:
# First, let's list the fields to choose appropriate columns for the summary
fields_cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "--list_fields",
]

print("Fetching column names for grouped summary test:\n")
fields_process = subprocess.run(fields_cmd, capture_output=True, text=True)
print(fields_process.stdout)

# Replace these with actual column names from your CSV
value_variable = "CG_epsilon"  # A numeric column
row_variable = "m"  # A categorical column for rows
column_variable = "MSCG_epsilon"  # A categorical column for columns

cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--add_grouped_summary",
    "--value_variable",
    value_variable,
    "--row_variable",
    row_variable,
    "--column_variable",
    column_variable,
    "--aggregation",
    "mean",  # Calculate mean values
    "--output_filename",
    "test_grouped_summary",
]

run_command(cmd, "Grouped summary tables")

### Test multiple options together

In [ ]:
cmd = [
    "python",
    SCRIPT_PATH,
    "-csv",
    MOCK_CSV_PATH,
    "-out",
    OUTPUT_DIRECTORY,
    "--list_fields",
    "--sample_rows",
    "3",
    "--uniqueness_report",
    "--field_statistics",
    "--output_format",
    "md",
    "--output_filename",
    "test_comprehensive_output",
]

run_command(cmd, "Combined analysis with multiple options")

### Test different aggregation methods

In [ ]:
# Test each available aggregation method
aggregation_methods = ["count", "min", "max", "mean"]

for agg_method in aggregation_methods:
    cmd = [
        "python",
        SCRIPT_PATH,
        "-csv",
        MOCK_CSV_PATH,
        "-out",
        OUTPUT_DIRECTORY,
        "--add_grouped_summary",
        "--value_variable",
        value_variable,
        "--row_variable",
        row_variable,
        "--column_variable",
        column_variable,
        "--aggregation",
        agg_method,
        "--output_filename",
        f"test_aggregation_{agg_method}",
    ]

    run_command(cmd, f"Grouped summary with {agg_method} aggregation")